# Skincare ML — uses `data/skincare_100.csv` (same as Lab2 seed)

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('..') / 'data'

In [ ]:
# optional: %matplotlib inline

In [ ]:
# Cell 3 — Data Loading
products = pd.read_csv(DATA_DIR / 'skincare_100.csv')
users = pd.read_csv(DATA_DIR / 'users.csv')
interactions = pd.read_csv(DATA_DIR / 'interactions.csv')

product_ids = set(products['Product_ID'].astype(str))
inter_col = next(
    (c for c in ['Product_ID', 'product_id', 'ProductID'] if c in interactions.columns),
    None,
)
if inter_col is None:
    raise ValueError(f'Product ID column not found in interactions. Columns: {list(interactions.columns)}')

interactions = interactions[interactions[inter_col].astype(str).isin(product_ids)].copy()

print('products:', products.shape)
print('users:', users.shape)
print('interactions (filtered):', interactions.shape)
print('\nSample product names (matches DB product.name):')
print(products['name'].head(10).to_string(index=False))

In [ ]:
# ... analysis cells 4–18 ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# Cell 19 — Feature Engineering / Merge

def pick_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    lower = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

user_id_col = pick_column(users, ['User_ID', 'user_id', 'UserID', 'id'])
product_id_col = pick_column(interactions, ['Product_ID', 'product_id', 'ProductID'])
product_id_products = pick_column(products, ['Product_ID', 'product_id', 'ProductID'])
budget_col = pick_column(users, ['Budget_Level', 'budget_level', 'Budget'])
price_col = pick_column(products, ['Price', 'price'])

severity_candidates = ['Acne', 'Dryness', 'Pigmentation', 'Aging', 'Sensitivity']
severity_cols = [c for c in severity_candidates if c in users.columns]
if not severity_cols:
    severity_cols = [
        c for c in users.columns
        if any(k in c.lower() for k in ['acne', 'dry', 'pigment', 'ag', 'sensit'])
    ]

users_feat = users.copy()
users_feat['total_skin_problem'] = users_feat[severity_cols].sum(axis=1)

df = interactions.merge(users_feat, on=user_id_col, how='left')
df = df.merge(
    products,
    left_on=product_id_col,
    right_on=product_id_products,
    how='left',
    suffixes=('', '_product'),
)

def budget_match(row):
    if pd.isna(row.get(price_col)) or pd.isna(row.get(budget_col)):
        return 0
    price = float(row[price_col])
    level = str(row[budget_col]).strip().lower()
    if level == 'low':
        return int(0 <= price <= 20)
    if level == 'medium':
        return int(20 < price <= 50)
    if level == 'high':
        return int(price > 50)
    return 0

df['budget_match'] = df.apply(budget_match, axis=1)

print('merged shape:', df.shape)
print(
    'total_skin_problem min/max:',
    df['total_skin_problem'].min(),
    df['total_skin_problem'].max(),
)
print('budget_match %:', round(100 * df['budget_match'].mean(), 2))

display_cols = ['name', 'category', 'total_skin_problem', 'budget_match']
display_cols = [c for c in display_cols if c in df.columns]
df[display_cols].head(3)